In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)


In [2]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')
curve_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,drn,Fn,pcr_plate,curve_idx
0,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,1,147103.828125,2517.961993,147103.83,AC00DB1I,24768
1,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,2,146864.656250,2278.790118,146864.66,AC00DB1I,24768
2,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,3,146410.109375,1824.243243,146410.11,AC00DB1I,24768
3,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,4,146188.328125,1602.461993,146188.33,AC00DB1I,24768
4,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,5,146078.421875,1492.555743,146078.42,AC00DB1I,24768


In [3]:
sample_info.head()

,sample_id,sample_barcode,pcr_plate,well_position,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file
0,S139294,LLIGI0000040467,AC00DBSX,B9,Clinical Sample,Negative,Negative,8/31/20,Submitted Sample,,,dbsx
1,S139295,LLIGI0000040896,AC00DBSX,F19,Clinical Sample,Negative,Negative,8/31/20,Submitted Sample,,,dbsx
2,S139296,LLIGI0000040750,AC00DBSX,B19,Clinical Sample,Negative,Negative,8/31/20,Submitted Sample,,,dbsx
3,S139299,LLIGI0000038073,AC00DBSX,L9,Clinical Sample,Negative,Negative,8/31/20,Submitted Sample,,,dbsx
4,S139163,200829023,AC00DBSX,G2,Clinical Sample,Negative,Negative,8/31/20,Submitted Sample,,,dbsx


In [4]:
join_df = (curve_df[~curve_df.curve_idx.isin([105249, 105633, 106017])] # exclude pooled samples
    .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
    .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))
join_df.head()

,well_position,target,dye,amp_score,cq,threshold,baseline_start,baseline_end,cycle_no,rn,drn,Fn,pcr_plate,curve_idx,sample_id,sample_barcode,sample_type,final_patient_result,current_sample_result,created_date,record_type,retest_sample_id_1,retest_sample_id_2,file,igi_call,thres_ct
0,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,1,147103.828125,2517.961993,147103.83,AC00DB1I,24768,S304772,POOL-SUR000065645-SUR000065635-SUR000065161-SU...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
1,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,2,146864.656250,2278.790118,146864.66,AC00DB1I,24768,S304772,POOL-SUR000065645-SUR000065635-SUR000065161-SU...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
2,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,3,146410.109375,1824.243243,146410.11,AC00DB1I,24768,S304772,POOL-SUR000065645-SUR000065635-SUR000065161-SU...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
3,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,4,146188.328125,1602.461993,146188.33,AC00DB1I,24768,S304772,POOL-SUR000065645-SUR000065635-SUR000065161-SU...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined
4,A1,S gene,ABY,0.0,Undetermined,8810.498,3,39,5,146078.421875,1492.555743,146078.42,AC00DB1I,24768,S304772,POOL-SUR000065645-SUR000065635-SUR000065161-SU...,Clinical Sample,Negative,Negative,44205,Pooled Sample,,,db1i,Negative,Undetermined


In [5]:
ms2_ids = join_df.loc[(join_df.sample_type == 'Clinical Sample') &
    (join_df.target.isin(['MS2'])) & 
        (join_df.cycle_no == 1) &
        (join_df.igi_call == 'Positive')].curve_idx.sample(100, random_state=1)

rnasep_ids = join_df.loc[(join_df.sample_type == 'Clinical Sample') & \
    (join_df.target.isin(['RnaseP'])) & \
        (join_df.cycle_no == 1) &
        (join_df.igi_call == 'Positive')].curve_idx.sample(100, random_state=1)
        

In [6]:
sgene_ids = join_df.loc[(join_df.sample_type == 'Positive Control (qPCR)') & \
    (join_df.target.isin(['S gene'])) & \
        (join_df.cycle_no == 1) &
        (join_df.igi_call == 'Positive')].curve_idx.sample(40, random_state=1).values
    
ngene_ids = join_df.loc[(join_df.sample_type == 'Positive Control (qPCR)') & \
    (join_df.target.isin(['N gene'])) & \
        (join_df.cycle_no == 1) &
        (join_df.igi_call == 'Positive')].curve_idx.sample(40, random_state=1).values
    
egene_ids = join_df.loc[(join_df.sample_type == 'Positive Control (qPCR)') & \
    (join_df.target.isin(['E gene'])) & \
        (join_df.cycle_no == 1) &
        (join_df.igi_call == 'Positive')].curve_idx.sample(40, random_state=1).values

In [7]:
neg_ids = join_df.loc[(join_df.sample_type == 'Negative Control (qPCR)') & 
            (join_df.igi_call == 'Negative') &
            (join_df.cycle_no == 1)].curve_idx.sample(100, random_state=1).values

In [9]:
valset = join_df.loc[join_df.curve_idx.isin(list(np.concatenate([rnasep_ids, ms2_ids, sgene_ids, ngene_ids, egene_ids, neg_ids]))),
            ['curve_idx','target','amp_score','cq','cycle_no','rn','drn','Fn','pcr_plate','sample_type','igi_call', 'threshold']]
valset[valset.cycle_no==1].groupby(['sample_type','target','igi_call']).curve_idx.count()

sample_type              target  igi_call
Clinical Sample          MS2     Positive    100
                         RnaseP  Positive    100
Negative Control (qPCR)  E gene  Negative     30
                         MS2     Negative      6
                         N gene  Negative     24
                         ORF1ab  Negative      6
                         RnaseP  Negative     26
                         S gene  Negative      8
Positive Control (qPCR)  E gene  Positive     40
                         N gene  Positive     40
                         S gene  Positive     40
Name: curve_idx, dtype: int64

In [14]:
valset.to_csv('./data/opt_valset.csv', index=False)

The whole pcr plate 'AC00GXWF', 'AC00DB6F', 'AC00H0OU' are invalid

Some samples inside pooled samples are also listed in sample info table. So we have multiple sample ids in the same well_position and pcr_plate.